# EHR Pipeline -- example notebook

End-to-end notebook for running the SMART-on-FHIR ETL pipeline against a
chunked CSV export from your clinical data warehouse.

## Required files in your working directory

Set `WORK_DIR` in Cell 2 to point at a directory containing:

```
<WORK_DIR>/
├── run_pipeline.py             pipeline code
├── dashboard.html              browser viewer (optional)
├── test_patients.txt           exclusion rules (optional)
├── uuid_mapping.csv            document_uuid -> patient_id bridge
└── CCDA and FHIR data/
    ├── ccda_chunks.csv         CCDA chunked export
    └── fhir_chunks.csv         FHIR chunked export
```

The notebook works in any Jupyter-compatible environment. There is an
optional Cell 1A that mounts Google Drive for users running in Colab.

Run cells in order. Pipeline takes a few minutes for a typical export.

## Cell 1 -- Install dependencies

In [ ]:
!pip install pandas pypdf openpyxl --quiet

## Cell 1A -- Mount Google Drive (Colab only)

Skip this cell if you're running in any other environment (local Jupyter,
JupyterLab, VS Code, etc.). Your inputs are read directly from `WORK_DIR`
in those environments.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    print('Drive mounted at /content/drive')
except ImportError:
    print('Not in Colab - skip this cell')

## Cell 2 -- Configure and stage inputs

Set `WORK_DIR` to your working directory. If your inputs are in a separate
location (e.g. a cloud-storage mount), set `SOURCE_DIR` to that path and the
cell copies them into `WORK_DIR`. Otherwise just set `SOURCE_DIR = None`.

In [ ]:
import os, shutil

# Working directory the pipeline reads from and writes to.
# Examples:
#   '/content/work/'                   <-- Colab + cloud storage source below
#   '/home/user/projects/registry/'    <-- local Jupyter
#   './'                               <-- current directory
WORK_DIR = './'

# Optional source directory to copy inputs from before running.
# Set to None if your inputs are already in WORK_DIR.
# Example: '/content/drive/MyDrive/your-project-folder/'
SOURCE_DIR = None

os.makedirs(WORK_DIR, exist_ok=True)

if SOURCE_DIR:
    print(f'Source folder contents: {SOURCE_DIR}')
    for fn in sorted(os.listdir(SOURCE_DIR)):
        p = os.path.join(SOURCE_DIR, fn)
        sz = os.path.getsize(p) if os.path.isfile(p) else '<dir>'
        print(f'  {fn}  ({sz})')

    def safe_copy(rel):
        src = os.path.join(SOURCE_DIR, rel)
        dst = os.path.join(WORK_DIR, rel)
        if not os.path.exists(src):
            print(f'  SKIP (missing): {rel}')
            return
        os.makedirs(os.path.dirname(dst) or '.', exist_ok=True)
        if os.path.isdir(src):
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f'  copied dir: {rel}')
        else:
            shutil.copy2(src, dst)
            print(f'  copied: {rel}  ({os.path.getsize(dst):,} bytes)')

    print(f'\nCopying inputs to {WORK_DIR} ...')
    safe_copy('CCDA and FHIR data/ccda_chunks.csv')
    safe_copy('CCDA and FHIR data/fhir_chunks.csv')
    safe_copy('uuid_mapping.csv')
    safe_copy('run_pipeline.py')
    safe_copy('test_patients.txt')
    safe_copy('dashboard.html')

print(f'\nWorking directory contents: {WORK_DIR}')
for root, dirs, files in os.walk(WORK_DIR):
    rel = os.path.relpath(root, WORK_DIR)
    if rel == '.': rel = ''
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f'  {os.path.join(rel, f)}  ({os.path.getsize(p):,} bytes)')

## Cell 3 -- Run the pipeline

All six stages: reassembly, format detection, CCDA + FHIR parsing with global
cross-bundle indexing, demographic fill from mapping, dedupe, enrichment,
test-patient filter, output writing.

If you're re-running and want to force a clean reassembly, delete
`<WORK_DIR>/ccda_assembled/` and `<WORK_DIR>/fhir_assembled/` before this cell.

In [ ]:
import sys, importlib, os

# Make sure run_pipeline.py reads from / writes to your WORK_DIR.
sys.path.insert(0, WORK_DIR)
if 'run_pipeline' in sys.modules:
    importlib.reload(sys.modules['run_pipeline'])
import run_pipeline

run_pipeline.BASE_DIR = WORK_DIR
run_pipeline.CCDA_CSV = os.path.join(WORK_DIR, 'CCDA and FHIR data/ccda_chunks.csv')
run_pipeline.FHIR_CSV = os.path.join(WORK_DIR, 'CCDA and FHIR data/fhir_chunks.csv')
run_pipeline.MAPPING_CANDIDATES = [os.path.join(WORK_DIR, 'uuid_mapping.csv')]
run_pipeline.EXCLUSION_CANDIDATES = [os.path.join(WORK_DIR, 'test_patients.txt')]

bundle = run_pipeline.main()

## Cell 4 -- Diagnostics

Sanity-check the bundle: counts per category, document linkage, key field
fill rates, PDF text extraction rate, and a few sample patients.

In [ ]:
from collections import Counter
import json, os

with open(os.path.join(WORK_DIR, 'dashboard_data.json')) as f:
    b = json.load(f)

print('=== HEADER ===')
print(f"Patients: {len(b.get('patients', []))}")
print(f"Documents: {len(b.get('documents', []))}")
total = sum(len(v) for k, v in b.items() if isinstance(v, list) and k != 'documents')
print(f"Records (excluding docs): {total:,}")

print('\n=== TAB COUNTS ===')
for k in ['encounters','problems','medications','procedures','labs','vitals','labs_vitals',
         'allergies','immunizations','careplans','diagnostic_reports','goals','notes']:
    rows = b.get(k, [])
    print(f"  {k:<24} {len(rows):>6,}")

print('\n=== DOCUMENT LINKAGE BY FORMAT ===')
fmt_total = Counter()
fmt_linked = Counter()
for d in b.get('documents', []):
    fmt = d.get('source_format', 'unknown')
    fmt_total[fmt] += 1
    if d.get('patient_id'):
        fmt_linked[fmt] += 1
for fmt, n in sorted(fmt_total.items()):
    pct = (fmt_linked[fmt] / n * 100) if n else 0
    print(f"  {fmt:<16} {n:>6,} total, {fmt_linked[fmt]:>6,} linked ({pct:5.1f}%)")

print('\n=== PDF TEXT EXTRACTION ===')
pdfs = [d for d in b.get('documents', []) if d.get('source_format') == 'pdf']
if pdfs:
    has_text = sum(1 for d in pdfs if d.get('plain_text') and not d['plain_text'].startswith('[PDF -'))
    no_text  = sum(1 for d in pdfs if d.get('plain_text', '').startswith('[PDF - no extractable text'))
    failed   = sum(1 for d in pdfs if d.get('plain_text', '').startswith('[PDF - extraction failed'))
    print(f"  PDFs with extracted text  : {has_text:>5,} / {len(pdfs):,}  ({has_text/len(pdfs)*100:5.1f}%)")
    print(f"  PDFs with no text         : {no_text:>5,}  (likely image-only scans - need OCR)")
    print(f"  PDFs that failed to parse : {failed:>5,}")
    if has_text > 0:
        sample = next(d for d in pdfs if d.get('plain_text') and not d['plain_text'].startswith('[PDF -'))
        snippet = sample['plain_text'][:200].replace('\n', ' ')
        print(f"\n  Sample PDF text: {snippet}...")
else:
    print('  no PDFs in bundle')

print('\n=== KEY FIELD FILL RATES ===')
for k in ['encounters','medications','problems','allergies','labs_vitals','notes']:
    rows = b.get(k, [])
    if not rows: continue
    n = len(rows)
    has_disp = sum(1 for r in rows if r.get('display_name') or r.get('section_title'))
    has_code = sum(1 for r in rows if r.get('code'))
    has_date = sum(1 for r in rows if r.get('effective_date') or r.get('start_date') or r.get('authored_on'))
    print(f"  {k} ({n:,}):  name={has_disp}/{n}  code={has_code}/{n}  date={has_date}/{n}")

print('\n=== SECTIONS (CCDA narratives) ===')
notes = b.get('notes', [])
if notes:
    with_text = sum(1 for r in notes if r.get('character_count', 0) > 0)
    total_chars = sum(r.get('character_count', 0) for r in notes)
    print(f"  {len(notes):,} sections, {with_text:,} with text, {total_chars:,} total chars")
    titles = Counter(r.get('section_title', '') for r in notes).most_common(5)
    print(f"  Top section titles: {titles}")

print('\n=== SAMPLE PATIENTS ===')
for p in b.get('patients', [])[:5]:
    print(f"  {p.get('first_name','')} {p.get('last_name','')} | MRN {p.get('mrn','-')} | DOB {p.get('dob','-')} | {p.get('num_documents',0)} docs")

## Cell 5 -- (Optional) Save outputs back to a destination folder

If you copied inputs from `SOURCE_DIR` in Cell 2, this copies the outputs
back to that location. Skip otherwise -- outputs are already in `WORK_DIR`.

In [ ]:
import shutil, os

if SOURCE_DIR:
    OUTPUTS = ['dashboard_data.json', 'dashboard_data.xlsx',
               'dashboard_data.prefilter.json', 'pipeline_run.log']
    for fn in OUTPUTS:
        src = os.path.join(WORK_DIR, fn)
        dst = os.path.join(SOURCE_DIR, fn)
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f'  saved: {fn} ({os.path.getsize(src):,} bytes)')
        else:
            print(f'  SKIP (missing): {fn}')

    csv_src = os.path.join(WORK_DIR, 'csv_exports')
    csv_dst = os.path.join(SOURCE_DIR, 'csv_exports')
    if os.path.exists(csv_src):
        if os.path.exists(csv_dst): shutil.rmtree(csv_dst)
        shutil.copytree(csv_src, csv_dst)
        print(f'  saved: csv_exports/ ({len(os.listdir(csv_dst))} files)')
else:
    print('No SOURCE_DIR set - outputs are already in WORK_DIR.')

## Cell 6 -- (Colab only) Download outputs to your machine

Skip this cell if you're not in Colab.

In [ ]:
import os
try:
    from google.colab import files
    json_path = os.path.join(WORK_DIR, 'dashboard_data.json')
    html_path = os.path.join(WORK_DIR, 'dashboard.html')
    if os.path.exists(json_path):
        files.download(json_path)
    if os.path.exists(html_path):
        files.download(html_path)
except ImportError:
    print('Not in Colab - outputs are on disk in', WORK_DIR)